In [2]:
# Installation des dépendances
!pip install ultralytics deepface tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.3 MB/s eta 0:00:00


In [3]:
import os, cv2, numpy as np, pandas as pd, torch
from tqdm import tqdm
from deepface import DeepFace
from ultralytics import YOLO
from collections import defaultdict

# Charge le modèle de pose
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_pose = YOLO('yolov8n-pose.pt').to(device)

26-03-02 23:32:48 - Directory /root/.deepface has been created
26-03-02 23:32:48 - Directory /root/.deepface/weights has been created
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
def process_human_from_keyframes(video_id, image_paths):
    stats = []

    for img_path in image_paths:
        frame = cv2.imread(img_path)
        if frame is None: continue

        # --- YOLO-POSE (Détection corps et visage) ---
        results = yolo_pose(frame, imgsz=320, verbose=False, conf=0.3)[0]

        face_count = 0
        max_face_area = 0
        has_body = 0

        if results.keypoints is not None:
            # keypoints.data : [x, y, conf]
            for kpts in results.keypoints.data:
                has_body = 1
                # Points 0-4 = Nez, Yeux, Oreilles (Visage)
                face_pts = kpts[:5]
                # Si au moins un point du visage est détecté avec confiance > 0.5
                if torch.any(face_pts[:, 2] > 0.5):
                    face_count += 1
                    x_min, y_min = torch.min(face_pts[:, :2], dim=0)[0]
                    x_max, y_max = torch.max(face_pts[:, :2], dim=0)[0]
                    # Calcul de la surface occupée par le visage
                    area = (x_max - x_min) * (y_max - y_min) / (frame.shape[0] * frame.shape[1])
                    if area > max_face_area:
                        max_face_area = float(area)

        # --- DEEPFACE (Émotion) ---
        emotion = "none"
        if face_count > 0:
            try:
                # Analyse de l'émotion sur la frame entière
                analysis = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False, silent=True)
                emotion = analysis[0]['dominant_emotion']
            except:
                emotion = "unknown"

        stats.append((face_count, round(max_face_area, 4), emotion, has_body))

    if not stats: return None

    # Agrégation des résultats de toutes les frames de la vidéo
    face_counts = [s[0] for s in stats]
    face_areas = [s[1] for s in stats]
    emotions = [s[2] for s in stats]
    bodies = [s[3] for s in stats]

    # On prend l'émotion la plus fréquente (ou "none" si rien)
    valid_emotions = [e for e in emotions if e not in ["none", "unknown"]]
    dominant_emotion = max(set(valid_emotions), key=valid_emotions.count) if valid_emotions else "none"

    return {
        'video_id': video_id,
        'avg_face_count': np.mean(face_counts),
        'max_face_coverage': np.max(face_areas),
        'dominant_emotion': dominant_emotion,
        'body_presence_score': np.mean(bodies)
    }

In [5]:
from google.colab import drive

# 1. Montage du Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Chemins à adapter
KEYFRAMES_DIR = "/content/drive/MyDrive/hackathon/keyframes/"
OUTPUT_CSV = "/content/drive/MyDrive/hackathon/features_humain.csv"

# 1. Grouper les fichiers par vidéo_id
video_groups = defaultdict(list)
for f in os.listdir(KEYFRAMES_DIR):
    if f.lower().endswith(('.jpg', '.jpeg')):
        v_id = f.rsplit('_kf', 1)[0]
        video_groups[v_id].append(os.path.join(KEYFRAMES_DIR, f))

# 2. Exécution
results = []
print(f"👥 Analyse humaine sur {len(video_groups)} vidéos...")

for v_id, paths in tqdm(video_groups.items()):
    data = process_human_from_keyframes(v_id, paths)
    if data:
        results.append(data)

# 3. Création du CSV et Dummies pour les émotions
if results:
    df_human = pd.DataFrame(results)
    # Transformation de l'émotion (texte) en colonnes 0/1 (One-Hot Encoding)
    if 'dominant_emotion' in df_human.columns:
        df_human = pd.get_dummies(df_human, columns=['dominant_emotion'])

    df_human.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Terminé : {len(df_human)} vidéos traitées.")

👥 Analyse humaine sur 1686 vidéos...


100%|██████████| 1686/1686 [37:51<00:00,  1.35s/it]

✅ Terminé : 1686 vidéos traitées.
